# Clusters as Features — putting clustering back into a supervised pipeline

Clustering does not have to be the end of the analysis. A cluster assignment is information
about a row, and information about a row is a **feature**.

This is the bridge from unsupervised back to everything in the Supervised ML module: use
k-means as a *transformer* inside a pipeline, and let a supervised model use the result.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RSEED = 42

penguins = pd.read_csv("data/penguins.csv").dropna(subset=["sex"])
X = penguins.drop(columns="sex")
y = penguins["sex"]

## `KMeans` is a transformer

`KMeans` has a `.transform()` method as well as `.predict()`. It returns, for each row, its
**distance to every centroid** — so `n_clusters` new numeric columns.

Those distances say "this penguin is close to group 2 and far from group 0", which is a
genuinely different description of the row from its raw measurements. A linear model can use
it to express things it otherwise could not.

In [ ]:
numeric = make_column_selector(dtype_include="number")

baseline = make_pipeline(
    make_column_transformer(
        (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
        (make_pipeline(SimpleImputer(strategy="most_frequent"),
                       OneHotEncoder(handle_unknown="ignore")),
         make_column_selector(dtype_exclude="number")),
    ),
    LogisticRegression(max_iter=1000),
)

scores = cross_val_score(baseline, X, y, cv=5, scoring="accuracy")
print(f"baseline (raw features)      : {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
def with_clusters(n_clusters):
    """Same pipeline, with k-means distances appended to the numeric branch."""
    numeric_branch = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    return make_pipeline(
        make_column_transformer(
            (numeric_branch, numeric),
            (make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                           KMeans(n_clusters=n_clusters, n_init=10, random_state=RSEED)), numeric),
            (make_pipeline(SimpleImputer(strategy="most_frequent"),
                           OneHotEncoder(handle_unknown="ignore")),
             make_column_selector(dtype_exclude="number")),
        ),
        LogisticRegression(max_iter=1000),
    )


for k in [2, 4, 8, 16]:
    s = cross_val_score(with_clusters(k), X, y, cv=5, scoring="accuracy")
    print(f"with {k:>2} cluster-distance features : {s.mean():.4f} ± {s.std():.4f}")

Compare the intervals, not just the means — the habit from the Bias-Variance lesson.

```
baseline (raw features)           0.9039 +/- 0.0150
with  2 cluster-distance features 0.8948 +/- 0.0167
with  4 cluster-distance features 0.8919 +/- 0.0176
with  8 cluster-distance features 0.8949 +/- 0.0190
with 16 cluster-distance features 0.8948 +/- 0.0167
```

**The cluster features made it slightly worse, at every value of `k`.**

Sit with that before reading on. It is the honest result on this data, and more instructive
than a win would have been.

## Why it did not help here

The technique gives a linear model a way to express **local** structure — "penguins near this
centroid behave differently" — a curved statement assembled from linear parts. Same idea as
the `SplineTransformer` and `Nystroem` material in the Feature Engineering lesson, arrived at
from a different direction.

It pays off when there is local structure the raw features cannot express. Here there is not:
we are predicting **sex** from body measurements, and the k-means clusters are driven almost
entirely by **species** — the dominant source of variation in this data. The new columns
describe something real, and something the model was not asked about. Extra columns carrying
no signal for *this* target cost variance and return nothing.

That is the general shape of the risk. **Cluster features encode whatever structure dominates
the feature space, which need not be the structure your target cares about.**

In [ ]:
search_pipeline = with_clusters(4)
cluster_step = [name for name in search_pipeline.get_params() if name.endswith("kmeans__n_clusters")][0]

search = GridSearchCV(
    search_pipeline,
    param_grid={cluster_step: [2, 3, 4, 6, 8, 12, 16, 24]},
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)
search.fit(X, y)

print(f"parameter searched : {cluster_step}")
print(f"best n_clusters    : {search.best_params_[cluster_step]}")
print(f"best cv accuracy   : {search.best_score_:.4f}")

The search picked `n_clusters = 3` at 0.8949 — still below the 0.9039 baseline. **A search
will always return a winner, even when every candidate is worse than not doing it at all.**
That is why the baseline comparison comes first, and why the Hyperparameter Tuning lesson
insisted on it.

Remember too what that lesson said about `best_score_`: it is the maximum of eight noisy
estimates, so it is optimistic even as a description of the losing option.

## Summary

- `KMeans.transform()` returns distances to every centroid — `n_clusters` new features.
- Inside a supervised pipeline this lets a linear model express local structure, the same
  goal as splines and kernel approximation from a different angle.
- Inside a pipeline, `n_clusters` becomes an ordinary hyperparameter a search can choose — no
  silhouette argument required.
- **On this data it did not help**, because the clusters track species while the target was
  sex. Cluster features encode whatever dominates the feature space, which need not be what
  your target cares about.
- Always compare against the same pipeline *without* the cluster features, and compare
  intervals rather than means. A search returning a "best" option does not mean the technique
  earned its place.

> This closes the loop: unsupervised learning as a step *inside* the supervised workflow,
> rather than a separate activity.